# 🌱 SAD-ADUBO — Análise Exploratória de Dados (EDA)

**TCC — Bacharelado em Engenharia Agrícola | UEG — UnU Santa Helena de Goiás | 2026**

**Autor:** Eduardo Augusto de Oliveira Mendes  
**Orientadora:** Prof.ª Me. Pollyana Queiroz

---

Este notebook realiza a **Análise Exploratória de Dados (EDA)** do dataset *Fertilizer Recommendation Dataset* (Kaggle), que contém 10.000 registros com 19 features e 1 variável-alvo.

## Objetivos
1. Compreender a distribuição e natureza das variáveis
2. Identificar correlações entre features
3. Verificar balanceamento da variável-alvo
4. Detectar outliers e padrões relevantes
5. Subsidiar decisões de pré-processamento e modelagem

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configurações de exibição
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Bibliotecas carregadas com sucesso')

## 1. Carregamento do Dataset

In [ ]:
# Carregar o dataset
df = pd.read_csv('../data/fertilizer_recommendation.csv')

print(f'📊 Dimensões do dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'\n📋 Colunas ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
# Visualizar primeiras linhas
df.head(10)

In [ ]:
# Informações do dataset
df.info()

## 2. Visão Geral dos Dados

In [ ]:
# Definir grupos de variáveis
TARGET_COL = 'Recommended_Fertilizer'

CATEGORICAL_COLS = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
    'Season', 'Irrigation_Type', 'Previous_Crop', 'Region'
]

NUMERICAL_COLS = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
    'Electrical_Conductivity', 'Nitrogen_Level',
    'Phosphorus_Level', 'Potassium_Level',
    'Temperature', 'Humidity', 'Rainfall',
    'Fertilizer_Used_Last_Season', 'Yield_Last_Season'
]

print(f'🎯 Variável-alvo: {TARGET_COL}')
print(f'📂 Variáveis categóricas ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}')
print(f'📏 Variáveis numéricas ({len(NUMERICAL_COLS)}): {NUMERICAL_COLS}')
print(f'\n📊 Total: {len(CATEGORICAL_COLS) + len(NUMERICAL_COLS)} features + 1 alvo = {len(df.columns)} colunas')

In [ ]:
# Verificar valores nulos
null_counts = df.isnull().sum()
print('🔍 Valores nulos por coluna:')
print(null_counts.to_string())
print(f'\n✅ Total de valores nulos: {null_counts.sum()}')

In [ ]:
# Verificar dados duplicados
duplicatas = df.duplicated().sum()
print(f'🔄 Registros duplicados: {duplicatas}')
print(f'📊 Registros únicos: {len(df) - duplicatas}')

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df[NUMERICAL_COLS].describe().round(3)

## 3. Análise da Variável-Alvo

In [ ]:
# Distribuição da variável-alvo
class_counts = df[TARGET_COL].value_counts()
class_pct = df[TARGET_COL].value_counts(normalize=True) * 100

print('🎯 Distribuição dos fertilizantes recomendados:')
print('=' * 50)
for fert in class_counts.index:
    print(f'  {fert:25s} → {class_counts[fert]:5d} ({class_pct[fert]:5.1f}%)')
print('=' * 50)
print(f'  Total: {class_counts.sum():,}')

In [ ]:
# Gráfico de barras da variável-alvo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Countplot
colors = sns.color_palette('Set2', n_colors=len(class_counts))
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribuição dos Fertilizantes Recomendados', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fertilizante', fontsize=12)
axes[0].set_ylabel('Contagem', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{count}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, pctdistance=0.85)
centre_circle = plt.Circle((0, 0), 0.60, fc='white')
axes[1].add_artist(centre_circle)
axes[1].set_title('Proporção dos Fertilizantes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Análise de balanceamento
ratio = class_counts.max() / class_counts.min()
print(f'\n📊 Razão de desbalanceamento (maior/menor): {ratio:.1f}x')
if ratio > 5:
    print('⚠️ Desbalanceamento significativo — considerar técnicas de balanceamento')
else:
    print('✅ Desbalanceamento moderado — aceitável para treinamento')

## 4. Análise das Variáveis Numéricas

In [ ]:
# Histogramas de todas as variáveis numéricas
fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLS):
    ax = axes[i]
    ax.hist(df[col], bins=40, color='#2E86AB', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequência')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Média: {df[col].mean():.2f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Mediana: {df[col].median():.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de todas as variáveis numéricas
fig, axes = plt.subplots(4, 3, figsize=(18, 18))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLS):
    ax = axes[i]
    bp = ax.boxplot(df[col], patch_artist=True, widths=0.6)
    bp['boxes'][0].set_facecolor('#2E86AB')
    bp['boxes'][0].set_alpha(0.7)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel('Valor')
    
    # Calcular outliers (IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    ax.set_xlabel(f'Outliers: {outliers} ({outliers/len(df)*100:.1f}%)', fontsize=9)

plt.suptitle('Boxplots das Variáveis Numéricas (com contagem de outliers)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Detecção de outliers (método IQR)
print('📦 Análise de Outliers (Método IQR):')
print('=' * 65)
print(f'{"Variável":35s} {"Q1":>8s} {"Q3":>8s} {"IQR":>8s} {"Outliers":>10s}')
print('-' * 65)

for col in NUMERICAL_COLS:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = outliers / len(df) * 100
    flag = '⚠️' if pct > 5 else ''
    print(f'{col:35s} {Q1:8.2f} {Q3:8.2f} {IQR:8.2f} {outliers:6d} ({pct:4.1f}%) {flag}')

print('=' * 65)
print('\n📌 Decisão: Outliers serão PRESERVADOS pois representam cenários agronômicos reais.')

## 5. Análise das Variáveis Categóricas

In [ ]:
# Distribuição das variáveis categóricas
fig, axes = plt.subplots(3, 3, figsize=(20, 18))
axes = axes.flatten()

for i, col in enumerate(CATEGORICAL_COLS):
    ax = axes[i]
    counts = df[col].value_counts()
    colors = sns.color_palette('Set2', n_colors=len(counts))
    bars = ax.barh(counts.index, counts.values, color=colors, edgecolor='white')
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('Contagem')
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                f'{count}', ha='left', va='center', fontsize=9)

# Ocultar eixo extra
for j in range(len(CATEGORICAL_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Variáveis Categóricas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabela de frequência detalhada
print('📋 Frequência das Variáveis Categóricas:')
print('=' * 60)
for col in CATEGORICAL_COLS:
    print(f'\n🔹 {col} ({df[col].nunique()} categorias):')
    counts = df[col].value_counts()
    for cat, count in counts.items():
        pct = count / len(df) * 100
        print(f'   {cat:25s} → {count:5d} ({pct:5.1f}%)')

## 6. Análise de Correlação

In [ ]:
# Matriz de correlação (variáveis numéricas)
corr_matrix = df[NUMERICAL_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlação de Pearson'},
            ax=ax)
ax.set_title('Mapa de Correlação (Pearson) — Variáveis Numéricas', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Correlações mais fortes
print('🔗 Correlações mais fortes (|r| > 0.3):')
print('=' * 60)

correlations = []
for i in range(len(NUMERICAL_COLS)):
    for j in range(i + 1, len(NUMERICAL_COLS)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            correlations.append({
                'Var 1': NUMERICAL_COLS[i],
                'Var 2': NUMERICAL_COLS[j],
                'r': r
            })

if correlations:
    df_corr = pd.DataFrame(correlations).sort_values('r', ascending=False, key=abs)
    for _, row in df_corr.iterrows():
        direction = '↑' if row['r'] > 0 else '↓'
        strength = 'forte' if abs(row['r']) > 0.7 else 'moderada'
        print(f"  {direction} {row['Var 1']:30s} × {row['Var 2']:30s} → r = {row['r']:+.3f} ({strength})")
else:
    print('  Nenhuma correlação forte encontrada entre as variáveis numéricas.')
    print('  Isso é comum em datasets com variáveis heterogêneas (solo + clima + manejo).')

## 7. Análise Bivariada (Features × Alvo)

In [ ]:
# Boxplots das variáveis numéricas por fertilizante recomendado
fig, axes = plt.subplots(4, 3, figsize=(22, 24))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLS):
    ax = axes[i]
    df.boxplot(column=col, by=TARGET_COL, ax=ax, grid=False,
              patch_artist=True,
              boxprops=dict(facecolor='#2E86AB', alpha=0.6),
              medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Variáveis Numéricas por Fertilizante Recomendado', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Média das variáveis numéricas por fertilizante
means_by_fert = df.groupby(TARGET_COL)[NUMERICAL_COLS].mean()
print('📊 Média das variáveis numéricas por fertilizante:')
means_by_fert.round(2)

In [ ]:
# Distribuição de fertilizantes por tipo de solo
ct_soil = pd.crosstab(df['Soil_Type'], df[TARGET_COL], normalize='index') * 100

fig, ax = plt.subplots(figsize=(14, 6))
ct_soil.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Distribuição de Fertilizantes por Tipo de Solo (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Proporção (%)')
ax.set_xlabel('Tipo de Solo')
ax.legend(title='Fertilizante', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição de fertilizantes por tipo de cultura
ct_crop = pd.crosstab(df['Crop_Type'], df[TARGET_COL], normalize='index') * 100

fig, ax = plt.subplots(figsize=(14, 6))
ct_crop.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white')
ax.set_title('Distribuição de Fertilizantes por Tipo de Cultura (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Proporção (%)')
ax.set_xlabel('Tipo de Cultura')
ax.legend(title='Fertilizante', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Análise de NPK (Nitrogênio, Fósforo, Potássio)

In [ ]:
# Scatter plot 3D de N, P, K colorido pelo fertilizante
fig = px.scatter_3d(
    df, x='Nitrogen_Level', y='Phosphorus_Level', z='Potassium_Level',
    color=TARGET_COL,
    title='Relação entre N, P, K por Fertilizante Recomendado',
    labels={
        'Nitrogen_Level': 'Nitrogênio (mg/kg)',
        'Phosphorus_Level': 'Fósforo (mg/kg)',
        'Potassium_Level': 'Potássio (mg/kg)'
    },
    opacity=0.6,
    height=600
)
fig.show()

In [ ]:
# Pairplot de N, P, K
npk_cols = ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']
g = sns.pairplot(df[npk_cols + [TARGET_COL]], hue=TARGET_COL,
                 palette='Set2', diag_kind='kde',
                 plot_kws={'alpha': 0.4, 's': 15})
g.fig.suptitle('Pairplot — N, P, K por Fertilizante', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 9. Análise de Importância Preliminar (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Preparar dados para importância preliminar
df_encoded = df.copy()
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(df_encoded[TARGET_COL])

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS
X = df_encoded[FEATURE_COLS]

# Treinar Random Forest rápido
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y_encoded)

# Feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#2E86AB' if imp > importances.mean() else '#A0D2E7' for imp in importances.values]
ax.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax.axvline(importances.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Média: {importances.mean():.4f}')
ax.set_title('Importância das Variáveis (Random Forest Preliminar)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importância (Gini)')
ax.legend()
plt.tight_layout()
plt.show()

print('\n🏆 Top 5 variáveis mais importantes:')
for i, (feat, imp) in enumerate(importances.tail(5).iloc[::-1].items(), 1):
    print(f'  {i}. {feat}: {imp:.4f}')

## 10. Conclusões da EDA

### Principais Achados

1. **Dataset limpo:** Nenhum valor nulo encontrado nas 10.000 linhas
2. **Variável-alvo:** 7 classes de fertilizantes com desbalanceamento moderado (Urea e DAP dominantes)
3. **Correlações:** Correlações fracas entre a maioria das variáveis numéricas (esperado para dados heterogêneos de solo/clima/manejo)
4. **Outliers:** Presentes principalmente em Rainfall e Temperature — preservados por representarem cenários agronômicos reais
5. **Features mais importantes (RF preliminar):** Nitrogen_Level, Soil_pH e Crop_Type lideram

### Decisões de Pré-processamento

- **Encoding:** Label Encoding para variáveis categóricas
- **Scaling:** MinMaxScaler para variáveis numéricas (range [0, 1])
- **Outliers:** Preservados
- **Balanceamento:** Não será aplicado SMOTE (desbalanceamento moderado)
- **Split:** 80/20, stratified, random_state=42